1. Instalar e importar PySpark

In [1]:
#!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import time
import shutil
import os

2. Crear sesión de Spark

In [2]:
spark = SparkSession.builder \
    .appName("Benchmark_PySpark_Colab") \
    .getOrCreate()

print("Spark listo")

Spark listo


3. Crear DataFrame de varios millones de filas

In [3]:
n = 5_000_000  # puedes subir a 10_000_000 si quieres probar más volumen

df = spark.range(0, n) \
    .withColumn("grupo", (col("id") % 10)) \
    .withColumn("subgrupo", (col("id") % 100))

print("DataFrame creado")
df.show(5)

DataFrame creado
+---+-----+--------+
| id|grupo|subgrupo|
+---+-----+--------+
|  0|    0|       0|
|  1|    1|       1|
|  2|    2|       2|
|  3|    3|       3|
|  4|    4|       4|
+---+-----+--------+
only showing top 5 rows


4. Ejecutar count()

In [4]:
start = time.time()
total = df.count()
end = time.time()

print(f"Total de filas: {total}")
print(f"Tiempo count(): {round(end - start, 2)} segundos")

Total de filas: 5000000
Tiempo count(): 2.36 segundos


5. Ejecutar groupBy

In [5]:
start = time.time()
resultado = df.groupBy("grupo").count()
resultado_cache = resultado.cache()  # opcional, para reutilizar
resultado_cache.show()
end = time.time()

print(f"Tiempo groupBy().count(): {round(end - start, 2)} segundos")

+-----+------+
|grupo| count|
+-----+------+
|    0|500000|
|    7|500000|
|    6|500000|
|    9|500000|
|    5|500000|
|    1|500000|
|    3|500000|
|    8|500000|
|    2|500000|
|    4|500000|
+-----+------+

Tiempo groupBy().count(): 9.87 segundos


6. Guardar resultado en Parquet

In [6]:
output_path = "resultado_parquet"

# Eliminar carpeta si ya existe
if os.path.exists(output_path):
    shutil.rmtree(output_path)

resultado_cache.write.mode("overwrite").parquet(output_path)

print(f"Resultado guardado en: {output_path}")

Resultado guardado en: resultado_parquet


7. Leer Parquet para validar

In [7]:
df_parquet = spark.read.parquet(output_path)
print("Lectura del Parquet guardado:")
df_parquet.show()

Lectura del Parquet guardado:
+-----+------+
|grupo| count|
+-----+------+
|    0|500000|
|    8|500000|
|    3|500000|
|    6|500000|
|    1|500000|
|    4|500000|
|    2|500000|
|    5|500000|
|    9|500000|
|    7|500000|
+-----+------+



8. Cerrar Spark

In [8]:
spark.stop()
print("Proceso finalizado")

Proceso finalizado
